![Week 6 Day 1](../../assets/banners/w6d1.png)

# Week 6 Day 1 - Solution: A Leaderboard Endpoint

**The exercise:** add `GET /leaderboard/{exam_id}?n=3` with an `APIRouter` and a Pydantic response model, include it in the notebook's app, test it with `TestClient`, and answer with a friendly 404 when there are no reports yet.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline in 🧪 demo mode.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ No OpenAI API Key set. That's fine this week: every cell runs offline in 🧪 demo mode.")
    print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day1_solution"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

### The router
Three habits from `gradewise/api/main.py`: a **response model** (so the docs and the validation come for free), a **`safe_name()`** check on anything that becomes a lookup key, and a **`GradeWiseError` subclass** for the failure case, so the app's exception handler turns it into the same friendly JSON as every other error. Subclassing `DataNotFoundError` gives us a 404 from `status_for()` without touching it.

In [ ]:
from typing import Annotated

from fastapi import APIRouter, Query, Request
from pydantic import BaseModel

from gradewise.api.workspace import safe_name
from gradewise.db import list_reports
from gradewise.errors import DataNotFoundError


class NoGradesYetError(DataNotFoundError):
    title = "No grades yet"
    cause = "Nothing has been graded for this exam in this workspace."
    fix = "Start a batch with POST /batches, approve its reviews, then try again."
    help_ref = "week6/solutions/day1_SOLUTION.ipynb"


class LeaderboardEntry(BaseModel):
    rank: int
    student_id: str
    percentage: float
    grade_band: str


leaderboard_router = APIRouter(tags=["reports"])


@leaderboard_router.get("/leaderboard/{exam_id}", response_model=list[LeaderboardEntry])
def leaderboard(
    request: Request,
    exam_id: str,
    n: Annotated[int, Query(ge=1, le=50, description="How many students")] = 3,
    demo: Annotated[bool | None, Query()] = None,
) -> list[LeaderboardEntry]:
    """The top `n` students on an exam, best first."""
    use_demo = request.app.state.default_demo if demo is None else demo
    ws = request.app.state.workspaces[bool(use_demo)]  # 💡 the same workspace logic as main.workspace()
    reports = list_reports(safe_name(exam_id, "exam ID"), db=ws.db)
    if not reports:
        raise NoGradesYetError(f"No report cards for exam {exam_id!r} yet.")
    best = sorted(reports, key=lambda r: r.percentage, reverse=True)[:n]
    return [
        LeaderboardEntry(rank=i, student_id=r.student_id, percentage=r.percentage, grade_band=r.grade_band)
        for i, r in enumerate(best, start=1)
    ]


app.include_router(leaderboard_router)
print("✅ /leaderboard/{exam_id} added")

### Before grading: a friendly 404, and validation for free

In [ ]:
for path in ["/leaderboard/cs_midterm", "/leaderboard/cs_midterm?n=0", "/leaderboard/..secret"]:
    r = client.get(path)
    body = r.json()
    print(r.status_code, path, "→", body.get("error") or body.get("detail", [{}])[0].get("msg"))

### After grading: the leaderboard

In [ ]:
# 🧱 Grade three synthetic students (approving every flag), so there are report cards to work with
batch_id = client.post("/batches", data={"exam_id": "cs_midterm", "from_transcripts": "true", "students": "3"}).json()[
    "batch_id"
]
batch = wait_for(batch_id)
while batch["status"] == "paused":
    client.post(f"/batches/{batch_id}/review", json={"approved": True}).raise_for_status()
    batch = wait_for(batch_id)
print(f"{batch_id}: {batch['status']} · {[r['student_id'] for r in batch['reports']]}")

In [ ]:
r = client.get("/leaderboard/cs_midterm", params={"n": 2})
print(r.status_code)
show(r.json())
assert [e["rank"] for e in r.json()] == [1, 2]
assert r.json()[0]["percentage"] >= r.json()[1]["percentage"]
print("✅ it's in the OpenAPI docs too:", "/leaderboard/{exam_id}" in app.openapi()["paths"])

💡 **Why a router instead of editing `main.py`?** It keeps your feature in its own module, testable on its own, and `include_router()` is all it takes to plug it in. That's also how you'd split a growing API into `routes/exams.py`, `routes/batches.py` and so on.

🧪 **Taking it further:** a leaderboard is a sensitive thing to publish in a classroom. Who should be allowed to call it? This is a good place to practise a FastAPI **dependency** that checks a bearer token (`Depends(...)`), so only teachers can see it.